# Exercício 1

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
caminho_entrada = "/FileStore/dados/PORTES_2025.csv"
caminho_saida = "/FileStore/isa/PORTES_2025_delta"

In [0]:
df = spark.read.csv(caminho_entrada, header=True, inferSchema=True, sep=";", encoding="latin1")

In [0]:
df.display()

In [0]:
df.printSchema()

## Verificar nulos e espaços

In [0]:
df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).display()

In [0]:
df.groupBy("ESPECIE_ARMA").count().display()

## Substituir espaços por "não informado"

In [0]:
for c, t in df.dtypes:
    if t == "string":
        df = df.withColumn(
            c,
            F.when(
                F.col(c).isNull() | (F.trim(F.col(c)) == ""),
                F.lit("não informado")
            ).otherwise(F.trim(F.col(c)))
        )

In [0]:
df.groupBy("ESPECIE_ARMA").count().display()

## Substituir espaços por "não informado" - forma alternativa

In [0]:
# df = df.select([
#     F.trim(F.col(c)).alias(c) if t == "string" else F.col(c)
#     for c, t in df.dtypes
# ])

In [0]:
# df.groupBy("ESPECIE_ARMA").count().display()


In [0]:
# df = df.na.replace("", "não informado")


In [0]:
# df.groupBy("ESPECIE_ARMA").count().display()

## Salvar Delta Table

In [0]:
df.coalesce(1) \
  .write.format("delta") \
  .mode("overwrite") \
  .save(caminho_saida)

df_delta = spark.read.format("delta").load(caminho_saida)
df_delta.display()